# Chapter 3: MNIST From Scratch

Applying everything from notebooks 1 & 2 to a real dataset.

**What we'll do:**
1. Load real image data (MNIST handwritten digits)
2. Proper train/validation/test split
3. Build a neural network from scratch
4. Train with early stopping
5. Save and load the best model
6. Final evaluation on held-out test data

This notebook works in Google Colab - no local setup needed.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

# Check if GPU is available (Colab offers free GPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Part 1: Loading MNIST

MNIST = 70,000 handwritten digit images (28x28 pixels, grayscale)
- 60,000 labeled as "train" 
- 10,000 labeled as "test"

In [ ]:
# Transform: convert images to tensors and normalize to 0-1 range
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts PIL image to tensor, scales to 0-1
])

# Download MNIST (will cache after first download)
train_full = datasets.MNIST(
    root='./data', 
    train=True, 
    download=True, 
    transform=transform
)

test_dataset = datasets.MNIST(
    root='./data', 
    train=False, 
    download=True, 
    transform=transform
)

print(f"Full training set: {len(train_full)}")
print(f"Test set: {len(test_dataset)}")

## Part 2: Train/Validation/Test Split

**The three sets:**

| Set | Purpose | When to look at it |
|-----|---------|-------------------|
| **Train** | Model learns from this | Every batch |
| **Validation** | Tune hyperparameters, decide when to stop | After each epoch |
| **Test** | Final evaluation only | Once, at the very end |

**Why validation matters:**
- If you tune based on test accuracy, you're indirectly "training" on test data
- Validation is your scratch paper; test is the final exam you only take once

In [ ]:
# Split the training data into train and validation
# Common splits: 80/20, 90/10, or for MNIST 50k/10k

train_size = 50000
val_size = 10000

# Set seed for reproducibility
torch.manual_seed(42)
train_dataset, val_dataset = random_split(train_full, [train_size, val_size])

print(f"Training set:   {len(train_dataset):,} samples")
print(f"Validation set: {len(val_dataset):,} samples")
print(f"Test set:       {len(test_dataset):,} samples (untouched until final eval)")

## Part 3: Exploring the Data

In [ ]:
# Get one sample (random_split wraps the dataset, so indexing still works)
image, label = train_dataset[0]

print(f"Image type: {type(image)}")
print(f"Image shape: {image.shape}")
print(f"Label: {label}")
print(f"\nShape breakdown: (channels, height, width) = {image.shape}")
print(f"  - 1 channel (grayscale, not RGB)")
print(f"  - 28 pixels tall")
print(f"  - 28 pixels wide")
print(f"  - Total pixels: {28 * 28} = 784")

In [ ]:
# Visualize a grid of samples
fig, axes = plt.subplots(3, 6, figsize=(12, 6))

for idx, ax in enumerate(axes.flat):
    image, label = train_dataset[idx]
    ax.imshow(image.squeeze(), cmap='gray')
    ax.set_title(f"{label}")
    ax.axis('off')

plt.suptitle("Sample MNIST Images (Training Set)", fontsize=14)
plt.tight_layout()
plt.show()

## Part 4: DataLoaders (Batching)

DataLoader groups samples into batches for efficient training.

In [ ]:
batch_size = 64

# Training: shuffle to prevent learning order-based patterns
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Validation & Test: no shuffle needed (just evaluating)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Batch size: {batch_size}")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

In [ ]:
# Get one batch
images, labels = next(iter(train_loader))

print(f"Batch of images shape: {images.shape}")
print(f"Batch of labels shape: {labels.shape}")
print(f"\nBreakdown: (batch_size, channels, height, width)")

## Part 5: Building the Model

Architecture:
```
Input (784 pixels) → Linear(784→128) → ReLU → Linear(128→64) → ReLU → Linear(64→10)
```

- 784 inputs (28x28 image flattened)
- Two hidden layers (128 and 64 neurons)
- 10 outputs (one per digit 0-9)

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers
        self.flatten = nn.Flatten()  # (batch, 1, 28, 28) → (batch, 784)
        self.layer1 = nn.Linear(784, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 10)
    
    def forward(self, x):
        # Define how data flows through
        x = self.flatten(x)        # Flatten image to 1D
        x = F.relu(self.layer1(x)) # Linear → ReLU
        x = F.relu(self.layer2(x)) # Linear → ReLU
        x = self.layer3(x)         # Final Linear (no activation - done by loss function)
        return x

model = MNISTNet().to(device)
print(model)

In [ ]:
# Count parameters (weights + biases)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
print(f"\nBreakdown:")
print(f"  Layer 1: 784 × 128 + 128 = {784*128 + 128:,}")
print(f"  Layer 2: 128 × 64 + 64 = {128*64 + 64:,}")
print(f"  Layer 3: 64 × 10 + 10 = {64*10 + 10:,}")

## Part 6: Loss Function and Optimizer

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("Loss function: CrossEntropyLoss")
print("  - Standard for multi-class classification")
print("  - Takes raw scores (logits) and class indices")
print(f"\nOptimizer: SGD with learning rate = 0.1")

## Part 7: Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """Train for one complete pass through the training data."""
    model.train()  # Set to training mode (enables dropout, etc.)
    total_loss = 0
    correct = 0
    total = 0
    
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        output = model(images)
        loss = loss_fn(output, labels)
        
        # Backward pass
        optimizer.zero_grad()  # Clear old gradients
        loss.backward()        # Compute new gradients
        optimizer.step()       # Update weights
        
        # Track metrics
        total_loss += loss.item()
        predicted = output.argmax(dim=1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

In [ ]:
def evaluate(model, loader, loss_fn, device):
    """Evaluate on validation or test data (no gradient updates)."""
    model.eval()  # Set to evaluation mode
    total_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():  # Don't track gradients (faster, less memory)
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            
            output = model(images)
            loss = loss_fn(output, labels)
            
            total_loss += loss.item()
            predicted = output.argmax(dim=1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

## Part 8: Training Loop with Early Stopping

**Early stopping:** Stop training when validation loss stops improving.

Why?
- Training loss will keep going down (memorizing)
- Validation loss starts going UP when overfitting begins
- We save the best model and stop when we're not improving

In [ ]:
# Training configuration
n_epochs = 20              # Maximum epochs to train
patience = 3               # Stop if no improvement for this many epochs
best_val_loss = float('inf')
epochs_without_improvement = 0
best_model_path = Path('./best_model.pth')

# Track history
history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

In [ ]:
# Reset model for fresh training
model = MNISTNet().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

print("Training with early stopping...")
print(f"Will stop if validation loss doesn't improve for {patience} epochs.\n")
print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>10} | {'Val Acc':>9} | {'Status':>10}")
print("-" * 75)

for epoch in range(n_epochs):
    # Train
    train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
    
    # Validate
    val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
    
    # Record history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Check for improvement
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        # Save the best model
        torch.save(model.state_dict(), best_model_path)
        status = "✓ Saved"
    else:
        epochs_without_improvement += 1
        status = f"No improve ({epochs_without_improvement}/{patience})"
    
    print(f"{epoch+1:>5} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>10.4f} | {val_acc:>8.2%} | {status:>10}")
    
    # Early stopping check
    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        break

print(f"\nBest validation loss: {best_val_loss:.4f}")
print(f"Model saved to: {best_model_path}")

In [ ]:
# Plot training progress
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs_run = len(history['train_loss'])

# Loss
ax1.plot(range(1, epochs_run+1), history['train_loss'], 'b-o', label='Train', markersize=4)
ax1.plot(range(1, epochs_run+1), history['val_loss'], 'r-o', label='Validation', markersize=4)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy
ax2.plot(range(1, epochs_run+1), history['train_acc'], 'b-o', label='Train', markersize=4)
ax2.plot(range(1, epochs_run+1), history['val_acc'], 'r-o', label='Validation', markersize=4)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Over Training')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

plt.tight_layout()
plt.show()

print("Watch the gap between train and validation - if it grows, that's overfitting!")

## Part 9: Load Best Model and Final Test Evaluation

**Important:** We only look at test data ONCE - right now.

This is the "final exam" - it tells us how well the model generalizes to truly unseen data.

In [ ]:
# Load the best model (lowest validation loss)
model.load_state_dict(torch.load(best_model_path))
print(f"Loaded best model from: {best_model_path}\n")

# Final evaluation on TEST set (first and only time!)
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)

print("=" * 50)
print("FINAL TEST RESULTS (first time seeing this data!)")
print("=" * 50)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.2%}")
print("=" * 50)

In [ ]:
# Compare all three sets
train_loss_final, train_acc_final = evaluate(model, train_loader, loss_fn, device)
val_loss_final, val_acc_final = evaluate(model, val_loader, loss_fn, device)

print("\nComparison of all three sets:")
print(f"{'Set':<12} | {'Loss':>8} | {'Accuracy':>10}")
print("-" * 38)
print(f"{'Train':<12} | {train_loss_final:>8.4f} | {train_acc_final:>10.2%}")
print(f"{'Validation':<12} | {val_loss_final:>8.4f} | {val_acc_final:>10.2%}")
print(f"{'Test':<12} | {test_loss:>8.4f} | {test_acc:>10.2%}")

print("\nIf test ≈ validation, our validation set was a good proxy!")
print("If train >> test, we're overfitting.")

## Part 10: Visualizing Predictions

In [ ]:
def predict_and_show(model, dataset, indices, device):
    """Show predictions for specific images."""
    model.eval()
    n = len(indices)
    fig, axes = plt.subplots(2, n, figsize=(2.5*n, 5))
    
    for i, idx in enumerate(indices):
        image, true_label = dataset[idx]
        
        # Get prediction
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(device))
            probs = F.softmax(output, dim=1)[0]
            pred_label = output.argmax(dim=1).item()
            confidence = probs[pred_label].item()
        
        # Show image
        axes[0, i].imshow(image.squeeze(), cmap='gray')
        color = 'green' if pred_label == true_label else 'red'
        axes[0, i].set_title(f"True: {true_label}\nPred: {pred_label} ({confidence:.0%})", color=color)
        axes[0, i].axis('off')
        
        # Show confidence bars
        colors = ['green' if j == true_label else 'gray' for j in range(10)]
        axes[1, i].barh(range(10), probs.cpu().numpy(), color=colors)
        axes[1, i].set_yticks(range(10))
        axes[1, i].set_xlim(0, 1)
        axes[1, i].set_xlabel('Confidence')
        if i == 0:
            axes[1, i].set_ylabel('Digit')
    
    plt.tight_layout()
    plt.show()

# Show some test predictions
print("Sample predictions from TEST set:")
predict_and_show(model, test_dataset, [0, 1, 2, 3, 4], device)

In [ ]:
# Find some mistakes
def find_mistakes(model, dataset, device, n=5):
    """Find misclassified images."""
    model.eval()
    mistakes = []
    
    for idx in range(len(dataset)):
        image, true_label = dataset[idx]
        
        with torch.no_grad():
            output = model(image.unsqueeze(0).to(device))
            pred_label = output.argmax(dim=1).item()
        
        if pred_label != true_label:
            mistakes.append(idx)
            if len(mistakes) >= n:
                break
    
    return mistakes

mistakes = find_mistakes(model, test_dataset, device, n=5)
print(f"\nMistakes (the model got these wrong):")
predict_and_show(model, test_dataset, mistakes, device)

## Part 11: Understanding What the Model Learned

In [ ]:
# Visualize first layer weights
weights = model.layer1.weight.data.cpu()

fig, axes = plt.subplots(4, 8, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    neuron_weights = weights[i].reshape(28, 28)
    ax.imshow(neuron_weights, cmap='RdBu', vmin=-0.5, vmax=0.5)
    ax.axis('off')
    ax.set_title(f'N{i}')

plt.suptitle('First Layer Weights: What each neuron "looks for"', fontsize=14)
plt.tight_layout()
plt.show()

print("Red = positive (looks for ink), Blue = negative (looks for no ink)")

## Part 12: Saving and Loading Models

Two ways to save:
1. **State dict only** (recommended) - just the weights
2. **Entire model** - weights + architecture

In [ ]:
# Method 1: Save state dict (recommended)
torch.save(model.state_dict(), 'model_weights.pth')
print("Saved: model_weights.pth")

# To load:
# model = MNISTNet()  # Create architecture
# model.load_state_dict(torch.load('model_weights.pth'))

# Method 2: Save entire model
torch.save(model, 'model_full.pth')
print("Saved: model_full.pth")

# To load:
# model = torch.load('model_full.pth')

print("\nMethod 1 is preferred - more portable and explicit.")

In [ ]:
# Save everything needed to resume training
checkpoint = {
    'epoch': len(history['train_loss']),
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'train_loss': history['train_loss'],
    'val_loss': history['val_loss'],
    'best_val_loss': best_val_loss,
}
torch.save(checkpoint, 'checkpoint.pth')
print("Saved full checkpoint (can resume training from here)")

## Part 13: Experimenting with Learning Rate

In [ ]:
def quick_train(lr, epochs=5):
    """Train a fresh model with given learning rate."""
    model = MNISTNet().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    
    train_losses = []
    val_losses = []
    
    for epoch in range(epochs):
        train_loss, _ = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        val_loss, _ = evaluate(model, val_loader, loss_fn, device)
        train_losses.append(train_loss)
        val_losses.append(val_loss)
    
    _, val_acc = evaluate(model, val_loader, loss_fn, device)
    return train_losses, val_losses, val_acc

# Compare learning rates
learning_rates = [0.001, 0.01, 0.1, 0.5]
results = {}

print("Testing different learning rates...\n")
for lr in learning_rates:
    train_losses, val_losses, val_acc = quick_train(lr)
    results[lr] = (train_losses, val_losses, val_acc)
    print(f"LR = {lr}: Final val accuracy = {val_acc:.2%}")

In [ ]:
# Plot learning rate comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for lr, (train_losses, val_losses, val_acc) in results.items():
    ax1.plot(train_losses, label=f'LR={lr}', marker='o', markersize=4)
    ax2.plot(val_losses, label=f'LR={lr} (acc={val_acc:.1%})', marker='o', markersize=4)

ax1.set_xlabel('Epoch')
ax1.set_ylabel('Training Loss')
ax1.set_title('Training Loss by Learning Rate')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.set_xlabel('Epoch')
ax2.set_ylabel('Validation Loss')
ax2.set_title('Validation Loss by Learning Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 14: Comparing Simple vs Deep Model

In [ ]:
# Simplest model: just one linear layer
simple_model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(784, 10)
).to(device)

simple_optimizer = torch.optim.SGD(simple_model.parameters(), lr=0.1)

print("Simple model (no hidden layers):")
print(f"Parameters: {sum(p.numel() for p in simple_model.parameters()):,}")
print(f"\nvs Deep model: {total_params:,} parameters")

In [ ]:
# Train the simple model
print("Training simple model...\n")

simple_history = {'train_acc': [], 'val_acc': []}

for epoch in range(10):
    train_loss, train_acc = train_one_epoch(simple_model, train_loader, loss_fn, simple_optimizer, device)
    val_loss, val_acc = evaluate(simple_model, val_loader, loss_fn, device)
    simple_history['train_acc'].append(train_acc)
    simple_history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1}: Train acc = {train_acc:.2%}, Val acc = {val_acc:.2%}")

# Test both models
_, simple_test_acc = evaluate(simple_model, test_loader, loss_fn, device)
_, deep_test_acc = evaluate(model, test_loader, loss_fn, device)

print(f"\n{'Model':<15} | {'Test Accuracy':>12} | {'Parameters':>12}")
print("-" * 45)
print(f"{'Simple (1 layer)':<15} | {simple_test_acc:>12.2%} | {sum(p.numel() for p in simple_model.parameters()):>12,}")
print(f"{'Deep (3 layers)':<15} | {deep_test_acc:>12.2%} | {total_params:>12,}")

In [ ]:
# Visualize simple model weights (what each digit "looks for")
simple_weights = simple_model[1].weight.data.cpu()

fig, axes = plt.subplots(2, 5, figsize=(15, 6))

for i, ax in enumerate(axes.flat):
    weights_img = simple_weights[i].reshape(28, 28)
    ax.imshow(weights_img, cmap='RdBu')
    ax.set_title(f'Digit {i}')
    ax.axis('off')

plt.suptitle('Simple Model: What each digit output "looks for"', fontsize=14)
plt.tight_layout()
plt.show()

print("You can see templates of each digit!")
print("Red = expects ink, Blue = expects no ink")

## Key Takeaways

### Data Splits
1. **Train** → model learns from this (every batch)
2. **Validation** → tune hyperparameters, early stopping (every epoch)
3. **Test** → final evaluation only (ONCE at the end)

### Training Best Practices
4. **Early stopping** prevents overfitting - stop when validation stops improving
5. **Save best model** based on validation loss, not training loss
6. **Checkpoints** let you resume training later

### Model Insights
7. **More layers** = more capacity to learn complex patterns
8. **Learning rate** matters - too small is slow, too big overshoots
9. **Gap between train/val** indicates overfitting
10. **Weights are interpretable** in simple models - you can see what they detect

## Exercises

1. **Different architecture**: Try 256→128→64→10. Does it help?

2. **Dropout**: Add `nn.Dropout(0.2)` after ReLU layers. Does it reduce overfitting?

3. **Different optimizer**: Replace SGD with `torch.optim.Adam(model.parameters(), lr=0.001)`

4. **Data augmentation**: Add random rotations/shifts to training data

5. **Binary classification**: Filter dataset to only 3s and 7s. Compare to multi-class.

## Cleanup

In [ ]:
# Optional: remove saved files
# import os
# for f in ['best_model.pth', 'model_weights.pth', 'model_full.pth', 'checkpoint.pth']:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"Removed {f}")